# Introduction to GPU Programming with Triton

This notebook is an **introduction to Triton**, a Python-like language for writing high-performance GPU kernels, and how it relates to **CUDA programming concepts** like threads and blocks.  

---

## CUDA Basics

When programming GPUs with **CUDA**, there are a few key concepts to understand:

### Threads
- The smallest unit of execution on a GPU.
- Each thread computes a single part of a problem, like one element of a matrix.
- Think of a thread as a “worker” that performs a tiny task.

### Blocks
- Threads are grouped into **blocks**.
- Threads in a block can share **fast shared memory** and synchronize with each other.
- Each block computes a portion of the output, like a small tile of a matrix.

### Grids
- Blocks are organized into a **grid** to cover the entire problem.
- Together, all blocks compute the full output.


## What is a CUDA Thread?

In CUDA programming:

- A **thread** is the **smallest unit of execution on the GPU**.
- Each thread executes the **same kernel code**, but usually operates on **different data**.

**Thread indexing:**

- `threadIdx` — the thread’s index **within its block**.
- `blockIdx` — the block’s index **within the grid**.
- `blockDim` — the number of threads **per block**.
- To get the **global index** of a thread (across all blocks):


In [1]:
import numpy as np
from numba import cuda


@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Get the global thread index
    idx = cuda.grid(1)

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)  # decreasing values
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads and blocks
threads_per_block = 8
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## What is a CUDA Block?

In CUDA programming:

- A **block** is a group of threads that execute together on a single **Streaming Multiprocessor (SM)** on the GPU.
- Threads in a block can:
  - Share **fast shared memory**.
  - Synchronize with each other using `cuda.syncthreads()`.
  
- Blocks are organized into a **grid** to cover larger datasets.

**Thread hierarchy:**

1. **Thread:** smallest unit of execution.
2. **Block:** a group of threads that can communicate via shared memory.
3. **Grid:** a group of blocks.

**Thread indexing within a block:**

- `threadIdx.x` — thread’s index **inside the block**.
- `blockIdx.x` — block’s index **inside the grid**.
- `blockDim.x` — number of threads **per block**.

**Global thread index formula:**


```global_idx = threadIdx.x + blockIdx.x * blockDim.x```



Each block is independent; threads in different blocks **cannot directly share shared memory**.


In [2]:
import numpy as np
from numba import cuda


# CUDA kernel for vector addition
@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Thread index inside block
    tx = cuda.threadIdx.x

    # Block index inside grid
    bx = cuda.blockIdx.x

    # Number of threads per block
    bdim = cuda.blockDim.x

    # Compute global thread index
    idx = tx + bx * bdim

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads per block and number of blocks
threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]
Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## Why Use Blocks in CUDA?

Using blocks is essential for **performance and scalability**:

1. **Shared Memory:**
   - Threads within the same block can share **fast shared memory**.
   - This avoids repeatedly reading the same data from **slower global memory**.

2. **Thread Cooperation:**
   - Threads in a block can **synchronize** using `cuda.syncthreads()`.
   - Useful for reductions, scans, or any operation where threads depend on each other.

3. **Tiling / Block-wise Computation:**
   - Large datasets can be **split into blocks (tiles)**.
   - Each block loads a chunk of data into shared memory, computes results locally, and writes back.
   - This **reduces global memory accesses** and improves performance.

We’ll demonstrate this with **tiled vector addition**, where each block loads its portion of two vectors into shared memory, adds them, and writes back the results.

## Tiled Vector Addition

Imagine we have:

- Vector A = [0, 1, 2, 3, 4, 5, 6, 7]  
- Vector B = [8, 7, 6, 5, 4, 3, 2, 1]  
- Threads per block = 4 → 2 blocks

---

### Step 1: Divide into Blocks (Tiles)

| Block | Thread IDs (tx) | Global Index (idx) |
|-------|----------------|------------------|
| 0     | 0,1,2,3       | 0,1,2,3         |
| 1     | 0,1,2,3       | 4,5,6,7         |

---

### Step 2: Load Tile into Shared Memory
```
Block 0 Shared Memory (sA, sB):
sA = [0,1,2,3]
sB = [8,7,6,5]

Block 1 Shared Memory (sA, sB):
sA = [4,5,6,7]
sB = [4,3,2,1]
```


- Each thread loads **one element** into shared memory.
- Threads in the block synchronize with `cuda.syncthreads()`.

---

### Step 3: Compute in Shared Memory
```
Block 0:
Thread 0: sA[0] + sB[0] = 0 + 8 = 8
Thread 1: sA[1] + sB[1] = 1 + 7 = 8
Thread 2: sA[2] + sB[2] = 2 + 6 = 8
Thread 3: sA[3] + sB[3] = 3 + 5 = 8

Block 1:
Thread 0: sA[0] + sB[0] = 4 + 4 = 8
Thread 1: sA[1] + sB[1] = 5 + 3 = 8
Thread 2: sA[2] + sB[2] = 6 + 2 = 8
Thread 3: sA[3] + sB[3] = 7 + 1 = 8
```

---

### Step 4: Write Back to Global Memory

- Each thread writes its computed value from **shared memory** to **global memory**.
- Multiple blocks work independently in parallel.


In [3]:
import numpy as np
from numba import cuda

MAX_SIZE = 32  # choose a safe upper bound for threads per block


# CUDA kernel: Tiled vector addition using "static" shared memory with flexible size
@cuda.jit
def vector_add_tiled(A: np.ndarray, B: np.ndarray, C: np.ndarray, sA_size: int, sB_size: int):
    # Declare max possible shared memory size
    sA = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)
    sB = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    idx = tx + bx * bdim

    # Only use the portion of shared memory according to passed sizes
    if idx < C.size and tx < sA_size:
        sA[tx] = A[idx]
    if idx < C.size and tx < sB_size:
        sB[tx] = B[idx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size and tx < sB_size:
        sA[tx] = sA[tx] + sB[tx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size:
        C[idx] = sA[tx]


# Data
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel with sizes for sA and sB
vector_add_tiled[blocks_per_grid, threads_per_block](A, B, C, threads_per_block, threads_per_block)

print("Vector C:", C)

Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [5]:
#play with code
# CUDA kernel: Tiled vector addition using "static" shared memory with flexible size
@cuda.jit
def vector_add_tiled(A: np.ndarray, B: np.ndarray, C: np.ndarray, sA_size: int, sB_size: int):
    # Declare max possible shared memory size
    sA = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)
    sB = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    idx = tx + bx * bdim

    # Only use the portion of shared memory according to passed sizes
    if idx < C.size and tx < sA_size:
        sA[tx] = A[idx]
    if idx < C.size and tx < sB_size:
        sB[tx] = B[idx]

    if idx < C.size and tx < sA_size and tx < sB_size:
        C[idx] = sA[tx] + sB[tx]


# Data
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel with sizes for sA and sB
vector_add_tiled[blocks_per_grid, threads_per_block](A, B, C, threads_per_block, threads_per_block)

print("Vector C:", C)

Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:934: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Triton Basics

Triton simplifies GPU programming by **removing much of the low-level thread/block management**.  

## Program
- In Triton, a **program** is like a “super-thread” that can operate on **a whole tile of data**.
- A program handles multiple elements at once using **vectorized operations**.
- Triton automatically maps these programs onto GPU threads and warps.

## Threads vs Programs
| Concept | CUDA | Triton |
|---------|------|--------|
| Smallest unit | Thread (1 element) | Program (1 tile of elements) |
| Shared memory | Threads in a block | Program uses fast memory implicitly |
| Mapping | ThreadIdx / BlockIdx | tl.program_id() |

## Tiled Vector Addition Using Triton

In Triton:

- A **kernel** is written as a Python function decorated with `@triton.jit`.
- Threads are organized in **blocks**, called **program IDs**.
- We can load **tiles of data into shared memory (using Triton’s `tl.load`)** for faster computation.
- Each thread (or “program”) handles a portion of the tile.

We will implement **vector addition C = A + B** using **tiled blocks**.

In [46]:
import torch
import triton
import triton.language as tl


# Triton kernel: vector addition with tiling
@triton.jit
def vector_add_tiled_kernel(
        A_ptr: torch.Tensor,
        B_ptr: torch.Tensor,
        C_ptr: torch.Tensor,
        N: tl.constexpr,
        BLOCK_SIZE: tl.constexpr
):
    # program ID = block index
    pid = tl.program_id(0)

    # compute start index of this block
    start = pid * BLOCK_SIZE

    # compute offsets for threads in the block
    offsets = start + tl.arange(0, BLOCK_SIZE)

    # Mask for threads that go out of bounds
    mask = offsets < N

    # Load from global memory
    a = tl.load(A_ptr + offsets, mask=mask)
    b = tl.load(B_ptr + offsets, mask=mask)

    # Compute addition
    c = a + b

    # Write back to global memory
    tl.store(C_ptr + offsets, c, mask=mask)


# Example vectors
N = 16
BLOCK_SIZE = 4
A = torch.arange(N, dtype=torch.float32, device='cuda')
B = torch.arange(N, 0, -1, dtype=torch.float32, device='cuda')
C = torch.zeros(N, dtype=torch.float32, device='cuda')

print("Vector A:", A)
print("Vector B:", B)

# Launch kernel
grid = (triton.cdiv(N, BLOCK_SIZE),)
vector_add_tiled_kernel[grid](A, B, C, N, BLOCK_SIZE=BLOCK_SIZE)

print("Result Vector C:", C)

Vector A: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')
Vector B: tensor([16., 15., 14., 13., 12., 11., 10.,  9.,  8.,  7.,  6.,  5.,  4.,  3.,
         2.,  1.], device='cuda:0')
Result Vector C: tensor([16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 16.], device='cuda:0')


## Why Tiled Vector Addition in Triton May Not Give Performance Boost

1. **Vector addition is memory-bound**:  
   - Each element is **read once from A and B, written once to C**.  
   - GPU arithmetic is cheap; most of the time is spent moving data.  

2. **Tiling only helps when data is reused**:  
   - In vector addition, each element is used **exactly once**.  
   - Loading tiles into shared memory (registers) **does not reduce global memory traffic**, because we still read each element once.  

3. **Overhead of tiling**:  
   - Extra instructions to compute offsets and masks may **slightly increase runtime** for small vectors.  

## Triton Matrix Multiplication Using Tiling


We are computing **C = A @ B**, where:

- A: (M x K)  
- B: (K x N)  
- C: (M x N)  

We use **tiled blocks** to exploit GPU parallelism and fast memory.

---

## 1. Block-level parallelism

- Each block computes a **tile of C** with size `BLOCK_M x BLOCK_N`.
- **Program IDs** identify each block:

```
pid_m = tl.program_id(0) # block row index
pid_n = tl.program_id(1) # block column index
```

- Block start indices:

```
m_start = pid_m * BLOCK_M
n_start = pid_n * BLOCK_N
```

- Example:  

```
BLOCK_M = 8, BLOCK_N = 8
pid_m=0, pid_n=0 -> computes C[0:8, 0:8]
pid_m=1, pid_n=0 -> computes C[8:16, 0:8]

```

---

## 2. Tiles and shared registers

- **accumulator**: each block creates a local `BLOCK_M x BLOCK_N` matrix in **registers**:


```acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)```


- This holds the **partial sum** of the block’s tile of C.
- Each element of the tile is computed by **multiple multiply-adds** over the K dimension.

---

## 3. Looping over K dimension in tiles

- The K dimension is usually too large to load fully into fast memory.
- Loop over K in **BLOCK_K chunks**:


```
for k_start in range(0, K, BLOCK_K):
    load A_tile: shape (BLOCK_M x BLOCK_K)
    load B_tile: shape (BLOCK_K x BLOCK_N)
    acc += A_tile @ B_tile
```

- Each tile is loaded from **global memory** into registers.  
- **Multiply-accumulate (tl.dot)** is performed in fast memory, **reducing global memory accesses**.

---

## 4. Offsets and masking

- Compute **global indices** for each tile:


```
offs_m = m_start + tl.arange(0, BLOCK_M)
offs_n = n_start + tl.arange(0, BLOCK_N)
offs_k = k_start + tl.arange(0, BLOCK_K)
```

- `tl.load` uses a **mask** to avoid out-of-bounds reads:


```mask=(offs_m[:, None] < M) & (offs_k[None, :] < K)```


- This ensures safety for the **last blocks** if M, N, or K are not multiples of tile size.

---

## 5. Multiply-accumulate (tl.dot)

- Compute:


```acc += tl.dot(A_tile, B_tile)```


- Each element in the tile of C is the **sum of products** along the K dimension.  
- After looping over all K tiles, `acc` contains the **final values** for this block’s C tile.

---

## 6. Write back to global memory

- After all K tiles are processed:


```tl.store(C_ptr + offs_m[:, None] * N + offs_n[None, :], acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))```


- Each block writes its tile of C to the correct location in global memory.  
- Multiple blocks write **independently in parallel**, covering the entire matrix.

---

## 7. Summary of Benefits

- **Tiling reduces global memory traffic**: each A and B element is loaded **once per tile** but used many times.  
- **Registers hold temporary results**, avoiding slow global memory for intermediate computations.  
- **Blocks operate independently** → full GPU parallelism.  
- **Masking ensures safety** for non-multiple dimensions.  

In [18]:
import torch
import triton
import triton.language as tl


@triton.jit
def matmul_kernel(
        A_ptr: torch.Tensor,  # tensor A
        B_ptr: torch.Tensor,  # tensor B
        C_ptr: torch.Tensor,  # tensor C
        M: tl.constexpr,  # number of rows in A/C
        N: tl.constexpr,  # number of columns in B/C
        K: tl.constexpr,  # number of columns in A / rows in B
        BLOCK_M: tl.constexpr,  # tile height
        BLOCK_N: tl.constexpr,  # tile width
        BLOCK_K: tl.constexpr  # tile depth along K
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    #pid_k = tl.program_id(2)

    start_m = pid_m * BLOCK_M
    start_n = pid_n * BLOCK_N
    #start_k = pid_k * BLOCK_K
    offs_m = start_m + tl.arange(0, BLOCK_M)
    offs_n = start_n + tl.arange(0, BLOCK_N)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for start_k in range(0, K, BLOCK_K):
      offs_k = start_k + tl.arange(0, BLOCK_K)

      a = tl.load(A_ptr + offs_m[:, None]*K + offs_k[None, :], mask=(offs_m[:, None] < M) & (offs_k[None, :] < K)) #[M,K]
      b = tl.load(B_ptr + offs_n[None, :] + offs_k[:, None]*N, mask=((offs_n[None, :] < N) & (offs_k[:, None] < K))) #[K,N]

      acc += tl.dot(a, b)
    tl.store(C_ptr + offs_m[:, None] * N + offs_n[None, :], acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


# Example usage
M, N, K = 16, 16, 16
BLOCK_M, BLOCK_N, BLOCK_K = 8, 8, 16  # BLOCK_K >= 16 required for tl.dot

A = torch.randn((M, K), dtype=torch.float32, device='cuda')
B = torch.randn((K, N), dtype=torch.float32, device='cuda')
C = torch.zeros((M, N), dtype=torch.float32, device='cuda')

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
matmul_kernel[grid](A, B, C, M, N, K, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K)

print("Matrix C:\n", C)

Matrix C:
 tensor([[ -0.4799,  -0.0484,   0.1072,  -1.8761,  -2.4730,   1.7780,  -1.8517,
          -3.0613,   3.2604,   2.2005,   4.4079,  -1.4237,   3.6768,  -2.6978,
          -1.2835,  -2.5933],
        [  2.4530,  -5.4774,   6.4540,   2.0567,   0.7924,   6.5096,   7.4317,
           1.6053,  -2.5716,  -0.5581,  -2.2157,  -3.5247,  -0.9952,   1.1407,
          -7.7835,  -0.5223],
        [  2.9444,  -1.5654,  -4.1647,  -1.3467,   1.4652,  -3.8066,   0.9460,
          -3.4004,  -1.3881,  -0.5395,  -6.0148,  -5.5970,  -3.1159,   4.2608,
          -0.5304,  -2.5977],
        [  4.4971,  -5.8817,   0.1921,   2.0428,   1.2123,   5.9814,   3.4711,
           5.4788,   7.9087,  -0.3761,  -0.7890,  -4.9401,   4.2783, -12.4216,
           2.5958,   2.4268],
        [ -0.3041,  -3.1024,   5.9105,  -0.7950,  -0.5264,   7.3136,   2.6199,
           1.7679,  -4.5578,  -1.5109,  -3.3669,   3.8073,   4.6368,   4.9523,
          -5.7578,   3.6299],
        [  2.0350,   6.5129,   4.5493,   4.0301, 

In [23]:
@triton.jit
def _seeded_dropout(
        x_ptr: torch.Tensor,
        output_ptr: torch.Tensor,
        n_elements: int,
        p: float,
        seed: int,
        BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    start = pid * BLOCK_SIZE
    offs = start + tl.arange(0, BLOCK_SIZE)
    x = tl.load(x_ptr + offs, mask = offs < n_elements)

    rnd = tl.rand(seed, offs)
    scale = 1.0 / (1.0 - p)
    out = tl.where(rnd>p, x * scale, 0.0)
    tl.store(output_ptr + offs, out, mask=offs < n_elements)


def seeded_dropout(x: torch.Tensor, p: float, seed: int) -> torch.Tensor:
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    _seeded_dropout[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


x = torch.randn(size=(10,)).cuda()
# Compare this to the baseline - dropout mask is never instantiated!
output = seeded_dropout(x, p=0.5, seed=123)

print("Input: ", x)
print("Out: ", output)

Input:  tensor([-0.2393, -0.7213, -0.3315, -0.9417,  1.9005, -0.8082, -1.0546,  1.3159,
         0.4801,  1.3075], device='cuda:0')
Out:  tensor([ 0.0000, -1.4426,  0.0000,  0.0000,  0.0000, -1.6164,  0.0000,  0.0000,
         0.9601,  2.6150], device='cuda:0')


# Homework

Реализуйте операторы на GELU. каждый оператор

# Task 1: Row-wise Matrix Sum

**Problem:**  

Implement a Triton kernel that computes **the sum of each row** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `row_sums` of shape `(M,)`  
- Hint:  
  - Use **tiles along the N dimension**  
  - Accumulate the sum in **registers**  
  - Handle partial tiles with **masking**

In [36]:
import torch
import triton
import triton.language as tl


@triton.jit
def row_sum_kernel(A_ptr: torch.Tensor,
                   row_sums_ptr: torch.Tensor,
                   M: int, N: int,
                   BLOCK_N: tl.constexpr):
    pid_n = tl.program_id(0) # one program per row
    row = pid_n
    acc = tl.zeros((), dtype = tl.float32)
    for start_n in range(0, N, BLOCK_N):
      offs_n = start_n + tl.arange(0, BLOCK_N)
      a = tl.load(A_ptr + row*N + offs_n, mask=(row<M) & (offs_n<N))
      acc += tl.sum(a, axis=0)
    tl.store(row_sums_ptr + row, acc, mask=row<M)



def run_kernel(A, BLOCK_N):
    M, N = A.shape
    row_sums = torch.zeros(M, device=A.device, dtype=A.dtype)
    grid = (M,)
    row_sum_kernel[grid](A, row_sums, M, N, BLOCK_N=BLOCK_N)
    return row_sums


def reference(A):
    return torch.sum(A, dim=1)


# --- Test 1: small fixed matrix ---
M, N, BLOCK_N = 4, 8, 4
A = torch.arange(M * N, dtype=torch.float32, device='cuda').reshape(M, N)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Matrix A:\n", A)
print("Kernel row sums:\n", row_sums)
print("Reference row sums:\n", row_sums_ref)
print("Match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 2: random matrix ---
M, N, BLOCK_N = 7, 13, 8  # not divisible by BLOCK_N
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Random test, match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_N = 64, 128, 32
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Large test, match?", torch.allclose(row_sums, row_sums_ref))



Matrix A:
 tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.]], device='cuda:0')
Kernel row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Reference row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 2: Column-wise Maximum

**Problem:**  

Implement a Triton kernel that computes **the maximum value in each column** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `col_max` of shape `(N,)`  
- Hint:  
  - Tile along the **M dimension** (rows)  
  - Use `tl.max` for reduction  
  - Mask out-of-bounds threads in the last tile

In [42]:
@triton.jit
def col_max_kernel(A_ptr: torch.Tensor,
                   col_max_ptr: torch.Tensor,
                   M: int, N: int,
                   BLOCK_M: tl.constexpr):
    pid = tl.program_id(0) # also idx of column
    acc = -float("inf")
    for start_m in range(0, M, BLOCK_M):
      offs_m = start_m + tl.arange(0, BLOCK_M)
      a = tl.load(A_ptr + offs_m * N + pid, mask = (pid<N) & (offs_m<M))
      acc = max(acc, tl.max(a, axis=0))
    tl.store(col_max_ptr + pid, acc, mask=pid<N)


# --- Helper functions ---
def run_col_max_kernel(A, BLOCK_M):
    M, N = A.shape
    col_max = torch.zeros(N, device=A.device, dtype=A.dtype)
    grid = (N,)
    col_max_kernel[grid](A, col_max, M, N, BLOCK_M=BLOCK_M)
    return col_max


def reference_col_max(A):
    return torch.max(A, dim=0).values


# --- Test 1: small fixed matrix ---
M, N, BLOCK_M = 6, 4, 4
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Matrix A:\n", A)
print("Kernel column max:\n", col_max)
print("Reference column max:\n", col_max_ref)
print("Match?", torch.allclose(col_max, col_max_ref))

# --- Test 2: random matrix not divisible by BLOCK_M ---
M, N, BLOCK_M = 7, 5, 2
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Random test, match?", torch.allclose(col_max, col_max_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_M = 64, 128, 16
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Large test, match?", torch.allclose(col_max, col_max_ref))

Matrix A:
 tensor([[ 0.6822, -1.4751,  0.6386, -1.2741],
        [-0.1868, -0.2037,  0.6499, -0.3372],
        [-0.8253, -0.2760,  0.5141, -0.3955],
        [-0.0809,  0.8266,  0.9530, -0.1304],
        [ 0.4051, -0.7676,  0.0963, -1.5042],
        [ 0.8953,  0.8834,  0.2278,  0.8456]], device='cuda:0')
Kernel column max:
 tensor([0.8953, 0.8834, 0.9530, 0.8456], device='cuda:0')
Reference column max:
 tensor([0.8953, 0.8834, 0.9530, 0.8456], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 3: GELU Activation

**Problem:**  

Implement the **GELU activation**:

$
\text{GELU}(x) = 0.5 \cdot x \cdot (1 + \tanh(\sqrt{2/\pi} \cdot (x + 0.044715 x^3)))
$

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Hint: Tile along the feature dimension and apply elementwise operations using `tl.pow`, `tl.tanh`, and broadcasting

In [78]:
import math

@triton.jit
def tanh_triton(x):
    e2x = tl.exp(2.0 * x)
    return (e2x - 1.0) / (e2x + 1.0)

@triton.jit
def gelu_kernel(X_ptr: torch.Tensor,
                Y_ptr: torch.Tensor,
                B: int, F: int,
                BLOCK_F: tl.constexpr):
    pid_b = tl.program_id(0)
    #acc = tl.zeros((F,), dtype = tl.float32)

    for start in range(0, F, BLOCK_F):
      offs = start + tl.arange(0, BLOCK_F)
      mask = (pid_b<B) & (offs < F)
      x = tl.load(X_ptr + offs +pid_b*F, mask=mask)
      #gelu = 0.5 * x * (1 + tanh_triton(math.sqrt(2.0/math.pi) * (x + 0.044715 * x * x * x)))
      gelu = 0.5 * x * (1 + tl.erf(x / math.sqrt(2.0)))
      tl.store(Y_ptr + offs +pid_b*F, gelu, mask=mask)


# --- Helper functions ---
def run_gelu_kernel(X, BLOCK_F):
    B, F = X.shape
    Y = torch.zeros_like(X)
    grid = (B,)
    gelu_kernel[grid](X, Y, B, F, BLOCK_F=BLOCK_F)
    return Y


def reference_gelu(X):
    return torch.nn.functional.gelu(X)


# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 4
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("X:\n", X)
print("Kernel GELU Y:\n", Y)
print("Reference GELU Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 4
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 32
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[ 0.7670, -0.5705, -0.1302, -1.8396, -1.8524, -0.1179, -1.2355, -0.1165],
        [ 2.5020,  0.3778, -1.0818, -1.1818,  0.4018,  0.9528,  1.2537, -1.0742]],
       device='cuda:0')
Kernel GELU Y:
 tensor([[ 0.5971, -0.1621, -0.0583, -0.0605, -0.0592, -0.0534, -0.1338, -0.0528],
        [ 2.4866,  0.2445, -0.1511, -0.1402,  0.2636,  0.7905,  1.1221, -0.1519]],
       device='cuda:0')
Reference GELU Y:
 tensor([[ 0.5971, -0.1621, -0.0583, -0.0605, -0.0592, -0.0534, -0.1338, -0.0528],
        [ 2.4866,  0.2445, -0.1511, -0.1402,  0.2636,  0.7905,  1.1221, -0.1519]],
       device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


## Task 4: Layer Normalization (Row-wise)

**Problem:**  

Implement **row-wise layer normalization** (without gamma/bamma).  

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Formula:  

$
Y[i,j] = \frac{X[i,j] - \text{mean}_i}{\sqrt{\text{var}_i + \epsilon}}
$

**Hints:**  
- Compute **row mean and variance in tiles**  
- Use `tl.sum` and `tl.pow` for accumulation

In [100]:
import torch
import triton
import triton.language as tl
import math
EPS = 1e-5

@triton.jit
def tl_mean(x, n):
  return tl.sum(x)/n

@triton.jit
def layer_norm_kernel(X_ptr: torch.Tensor,
                      Y_ptr: torch.Tensor,
                      B: int, F: int,
                      BLOCK_F: tl.constexpr,
                      eps):
    pid_b = tl.program_id(0)

    mean = 0
    _mean = tl.zeros([BLOCK_F], dtype=tl.float32)
    for start in range(0, F, BLOCK_F):
        offs = start + tl.arange(0, BLOCK_F)
        a = tl.load(X_ptr + offs + pid_b*F, mask=offs < F, other=0.).to(tl.float32)
        _mean += a
    mean = tl.sum(_mean, axis=0) / F

    _var = tl.zeros([BLOCK_F], dtype=tl.float32)
    for start in range(0, F, BLOCK_F):
        offs = start + tl.arange(0, BLOCK_F)
        x = tl.load(X_ptr + offs + pid_b*F, mask=offs < F, other=0.).to(tl.float32)
        x = tl.where(offs < F, x - mean, 0.)
        _var += x * x
    var = tl.sum(_var, axis=0) / F


    for start in range(0, F, BLOCK_F):
      offs = start + tl.arange(0, BLOCK_F)
      mask = (offs<F) & (pid_b<B)
      x = tl.load(X_ptr + offs + pid_b*F, mask=mask)
      out = (x - mean)/tl.sqrt(var+eps)
      tl.store(Y_ptr + offs + pid_b*F, out, mask=mask)
# --- Helper functions ---
def run_layer_norm_kernel(X, BLOCK_F):
    B, F = X.shape
    Y = torch.zeros_like(X)
    grid = (B,)
    layer_norm_kernel[grid](X, Y, B, F, BLOCK_F=BLOCK_F, eps=EPS)
    return Y


def reference_layer_norm(X, eps=EPS):
    mean = torch.mean(X, dim=1, keepdim=True)
    var = torch.var(X, dim=1, unbiased=False, keepdim=True)
    return (X - mean) / torch.sqrt(var + eps)


# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 4
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("X:\n", X)
print("Kernel LayerNorm Y:\n", Y)
print("Reference LayerNorm Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 4
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 32
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[ 0.9236,  1.4688, -0.9373, -1.1927, -0.0306, -2.0043,  0.7154,  1.0851],
        [ 0.7787,  0.8753,  0.4791,  1.9895, -0.7988, -1.8043,  0.0066, -0.9341]],
       device='cuda:0')
Kernel LayerNorm Y:
 tensor([[ 0.7840,  1.2486, -0.8016, -1.0193, -0.0290, -1.7108,  0.6066,  0.9216],
        [ 0.6216,  0.7068,  0.3573,  1.6896, -0.7698, -1.6568, -0.0594, -0.8892]],
       device='cuda:0')
Reference LayerNorm Y:
 tensor([[ 0.7840,  1.2486, -0.8016, -1.0193, -0.0290, -1.7108,  0.6066,  0.9216],
        [ 0.6216,  0.7068,  0.3573,  1.6896, -0.7698, -1.6568, -0.0594, -0.8892]],
       device='cuda:0')
Match? True
Random test, match? True
Large test, match? True
